In [ ]:
import pandas as pd
import re
import collections
import matplotlib.pyplot as plt
import numpy as np
from nltk.util import ngrams
from sklearn import preprocessing
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from itertools import chain
from random import randrange, uniform
import math
pd.set_option('display.max_columns', None)

In [ ]:
data = pd.read_csv("dns-traffic.20140614.txt",nrows=10000,sep="\t")
data.drop(data.columns[[0, 1, 2, 3, 4]], axis=1,inplace=True)
data.columns = ['UserId','Domain1','Domain']
print(data.head(20))

In [ ]:
for i in range(data['Domain'].shape[0]):
    if re.search("^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$",data['Domain'][i]) or re.search(":",data['Domain'][i]):
        temp = data['Domain1'][i]
        data.loc[i,('Domain')] = temp
    if re.search(r"\.$",data['Domain'][i]): 
        value = data['Domain'][i][:-1]
        data.loc[i,('Domain')] = value

data.drop(data.columns[1], axis=1,inplace=True)
print(data.head(20))

In [ ]:
whiteList = pd.read_csv('top-1m.csv',header=None,names=['Domain'])
print(whiteList.head(20))

In [ ]:
#DOMAIN FILTERING

#1.WhiteList filtering

tempList = whiteList.head(100).values
actualList=[]
for i in tempList:
    for j in i:
        actualList.append("[a-zA-Z0-9]*."+j)
        
reduceList=data.values
domainList=[]
dataList=[]
for i in reduceList:
    flag=0
    for j in actualList:
        if re.search(j,i[1]):
            flag=1
            break
    if flag==0:
        domainList.append(list(i)[1])
        dataList.append(i)

print("The number of remaining domain list is",len(domainList))

In [ ]:
#n-gram frequencies

#FOR WHITELIST
tokens = whiteList['Domain'].head(1000)
dictionary={}
probList=[]
for i in tokens:
    dumm = [i[j:j+2] for j in range(len(i)-1)]
    probList.append(dumm)
    bgs = ngrams(dumm,2)
    frequencies = collections.Counter(bgs)
    for j in frequencies:
        if j not in dictionary:
            dictionary[j]=frequencies[j]
        else:
            dictionary[j]+=frequencies[j]
            
print(collections.Counter(dictionary).most_common(5))

keys = list(dictionary.keys())
values = list(dictionary.values())

total_sum = sum(dictionary.values())
probabilities=[]
for i in probList:
    summation=0
    for j in i:
        subset_dict = [key for key in keys if j in key]
        for k in subset_dict:
            summation+=(dictionary[k]/total_sum)
    probabilities.append(summation)

res = '#'.join(map(str, keys))
keys = res.split('#')
number = preprocessing.LabelEncoder()
dumm = number.fit_transform(keys)

x = np.array(dumm)
y = np.array(values)

plt.scatter(x, y)
plt.show()

print("The first whiteList domain and its probability")
print(whiteList['Domain'].iloc[0]," ",probabilities[0])

In [ ]:

#FOR DOMAINLIST
tokens = domainList[:1000]
dictionary={}
probList=[]
for i in tokens:
    dumm = [i[j:j+2] for j in range(len(i)-1)]
    probList.append(dumm)
    bgs = ngrams(dumm,2)
    frequencies = collections.Counter(bgs)
    for j in frequencies:
        if j not in dictionary:
            dictionary[j]=frequencies[j]
        else:
            dictionary[j]+=frequencies[j]
            
print(collections.Counter(dictionary).most_common(5))

keys = list(dictionary.keys())
values = list(dictionary.values())

total_sum = sum(dictionary.values())
probabilities=[]
for i in probList:
    summation=0
    for j in i:
        subset_dict = [key for key in keys if j in key]
        for k in subset_dict:
            summation+=(dictionary[k]/total_sum)
    probabilities.append(summation)

res = '#'.join(map(str, keys))
keys = res.split('#')
number = preprocessing.LabelEncoder()
dumm = number.fit_transform(keys)

x = np.array(dumm)
y = np.array(values)

plt.scatter(x, y)
plt.show()

print("The first DGA domain and its probability")
print(domainList[0]," ",probabilities[0])

In [ ]:
#2. k-means clustering

domainsList = domainList[:1000]
clustList = number.fit_transform(domainsList)
trainList=[]
for i in range(len(clustList)):
    trainList.append([clustList[i],probabilities[i]])
training_set = np.array(trainList)

kmeans = KMeans(n_clusters=2,max_iter=300)
kmeans.fit(training_set)
labels = kmeans.labels_
labels=list(labels)


plt.scatter(training_set[:,0], training_set[:,1], c=labels)  
plt.show()

print("The count of non-benign domains is",labels.count(0))
print("The count of benign domains is",labels.count(1))

filteredOutput=[]
sideOutput=[]
for i in range(len(labels)):
    if labels[i] == 0:
        filteredOutput.append(domainList[i])
        sideOutput.append(dataList[i][0])

In [ ]:
#DOMAIN CLUSTERING

list1 = number.fit_transform(sideOutput)
list2 = number.fit_transform(filteredOutput)

inp = pd.DataFrame(list(zip(list1,list2)),columns=['Users','Domain'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(inp)
X_normalized = normalize(X_scaled) 
X_normalized = pd.DataFrame(X_normalized, columns=['Users','Domain'])

db_default = DBSCAN(eps = 0.00375, min_samples = 2).fit(X_normalized)
new_labels = db_default.labels_

print(new_labels)
new_labels = list(new_labels)
print("\nThe number of unique clusters  -",(len(collections.Counter(new_labels).keys())-1))
print("The number of noise values(-1) -",new_labels.count(-1))

plt.scatter(X_normalized['Users'], X_normalized['Domain'], c=new_labels)  
plt.show()

In [ ]:
new_dictionary={}
for i in range(len(new_labels)):
    if new_labels[i]!=-1:
        if new_labels[i] not in new_dictionary:
            new_dictionary[new_labels[i]] = [[sideOutput[i],filteredOutput[i]]]
        else:
            new_dictionary[new_labels[i]].append([sideOutput[i],filteredOutput[i]])

print("The list of clusters are ",end="")
print(list(new_dictionary.keys()))

In [ ]:
#COLLABORATIVE FILTERING

dgaDomains = pd.read_csv('dga_domains.csv',nrows=100)
dgaDomains.drop(['DGA_family','Type'], axis=1,inplace=True)
dgaDomains['Score'] = 1.0
print(dgaDomains.head())


In [ ]:
#Assign score values for clusters

for i in range(len(new_dictionary)):
    score = round(uniform(0.6, 0.8),2)
    for j in range(len(new_dictionary[i])):
        if new_dictionary[i][j][1] in list(dgaDomains['Domain']):
            new_dictionary[i][j].append(1.0)
        else:
            new_dictionary[i][j].append(score)

print(new_dictionary)

In [ ]:
#Finding scores for users

users=[]
for i in range(len(new_dictionary)):
    for j in range(len(new_dictionary[i])):
        if new_dictionary[i][j][0] not in users:
            users.append(new_dictionary[i][j][0])

maxi=[]
domain=[]
domainDict={}
for i in users:
    temp=0
    dom=""
    for j in range(len(new_dictionary)):
        for k in range(len(new_dictionary[j])):
            if new_dictionary[j][k][0] not in domainDict:
                domainDict[new_dictionary[j][k][0]] = [new_dictionary[j][k][1]]
            else:
                domainDict[new_dictionary[j][k][0]].append(new_dictionary[j][k][1])
            if new_dictionary[j][k][0] == i:
                if new_dictionary[j][k][2] > temp:
                    temp = new_dictionary[j][k][2]
                    dom = new_dictionary[j][k][1]
    
    maxi.append(temp)
    domain.append(dom)

    
from texttable import Texttable

tableList=[]
tableList.append(["User Ids","Scores","Domain"])

for i in range(len(users)):
    maxi[i]=round(pow(maxi[i],2),2)
    tableList.append([users[i],maxi[i],domain[i]])

print("\n\t\tScores Table")
table = Texttable()
table.add_rows(tableList)
print(table.draw())
print("\n")


In [ ]:
def cosine_similarity(c1, c2):
    terms = len(c1)
    dotprod = sum(c1[k] * c2[k] for k in range(terms))
    magA = math.sqrt(sum(c1[k]**2 for k in range(terms)))
    magB = math.sqrt(sum(c2[k]**2 for k in range(terms)))
    return dotprod / (magA * magB)

domainNames = domainDict[3600][:39] + domainDict[7200]
domainNames = number.fit_transform(domainNames)
user1 = domainNames[:39]
user2 = domainNames[39:]

print("The similarity between the user 3600 and user 7200 is ",end="")
print(cosine_similarity(user1,user2))